# Weaviate 1.19.0 to 1.33.1 Migration for Dify

This notebook guides you through migrating Dify knowledge bases from Weaviate 1.19.0 to 1.33.1 while preserving all embeddings.

## Overview
1. **Backup** data from Weaviate 1.19.0
2. **Upgrade** to Weaviate 1.33.1  
3. **Restore** backup
4. **Migrate** schema to fix vectorConfig
5. **Verify** in Dify UI


## Setup: Install Dependencies


In [5]:
%pip install -U weaviate-client requests


Note: you may need to restart the kernel to use updated packages.


In [21]:

import weaviate
from weaviate.classes.config import Configure, VectorDistances
import requests
import json
import time

# Configuration
WEAVIATE_HOST = "localhost"
WEAVIATE_PORT = 8080
WEAVIATE_GRPC_PORT = 50051
WEAVIATE_API_KEY = "WVF5YThaHlkYwhGUSmCRgsX3tD5ngdN8pkih"
BACKUP_ID = "dify-backup-v1-19"


---
## PART 1: Backup on Weaviate 1.19.0

**Before running these cells, make sure:**
1. You're on the OLD Dify version (commit with Weaviate 1.19.0)
2. You've added backup module to docker-compose.yaml:
   ```yaml
   weaviate:
     volumes:
       - ./volumes/weaviate:/var/lib/weaviate
       - ./volumes/weaviate_backups:/var/lib/weaviate/backups
     ports:
       - "8080:8080"
       - "50051:50051"
     environment:
       ENABLE_MODULES: backup-filesystem
       BACKUP_FILESYSTEM_PATH: /var/lib/weaviate/backups
   ```
3. Docker is running: `docker compose --profile weaviate up -d`
4. You've uploaded documents in Dify Knwledge base with **High Quality** indexing
5. Processing is complete (status shows "Completed")


### Step 1: Check Weaviate Connection and Backup Module


In [24]:
# Check if Weaviate is accessible
response = requests.get(f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/meta")

if response.status_code == 200:
    meta = response.json()
    print(f"Connected to Weaviate {meta.get('version')}")
    
    # Check if backup module is available
    if "backup-filesystem" in meta.get("modules", {}):
        print(f"Backup module is enabled")
        print(f"  Backup path: {meta['modules']['backup-filesystem']['backupsPath']}")
    else:
        print("WARNING: Backup module is NOT enabled!")
        print("  Add ENABLE_MODULES: backup-filesystem to docker-compose.yaml")
else:
    print(f"Cannot connect to Weaviate")
    print(f"  Make sure Docker is running and ports are exposed")


ConnectionError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /v1/meta (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x13124a240>: Failed to establish a new connection: [Errno 61] Connection refused'))

### Step 2: Create Backup (All Collections)


In [10]:
# Create backup of ALL collections (no need to list them manually)
backup_request = {
    "id": BACKUP_ID
}

response = requests.post(
    f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/backups/filesystem",
    json=backup_request,
    headers={"Authorization": f"Bearer {WEAVIATE_API_KEY}"}
)

if response.status_code in [200, 201]:
    result = response.json()
    print(f"Backup started: {result.get('id')}")
    print(f"  Status: {result.get('status')}")
    print(f"  Collections: {result.get('classes')}")
else:
    print(f"Backup failed: {response.text}")


Backup started: dify-backup-v1-19
  Status: STARTED
  Collections: ['Vector_index_47c65e1f_895e_4ff6_825c_8d2fce9cf011_Node']


### Step 3: Wait for Backup to Complete


In [11]:
# Poll backup status
print("Waiting for backup to complete...")

for i in range(30):  # Up to 30 seconds
    time.sleep(1)
    
    response = requests.get(
        f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/backups/filesystem/{BACKUP_ID}",
        headers={"Authorization": f"Bearer {WEAVIATE_API_KEY}"}
    )
    
    if response.status_code == 200:
        status_data = response.json()
        status = status_data.get('status')
        
        if status == 'SUCCESS':
            print(f"\nBackup completed successfully!")
            print(f"  Backup ID: {status_data.get('id')}")
            print(f"  Collections backed up: {status_data.get('classes')}")
            break
        elif status == 'FAILED':
            print(f"\nBackup failed!")
            print(json.dumps(status_data, indent=2))
            break
        elif i % 5 == 0:
            print(f"  Status: {status}...")


Waiting for backup to complete...

Backup completed successfully!
  Backup ID: dify-backup-v1-19
  Collections backed up: None


### Step 4: Verify Backup Files Created


In [ ]:
!ls -lh docker/volumes/weaviate_backups/dify-backup-v1-19/


---
## PART 2: Upgrade to Weaviate 1.33.1

**Manual Steps (run these in terminal):**

```bash
# 1. Stash your docker-compose changes
cd /Users/dhruvgorasiya/Documents/Weaviate/Integrations/dify
git stash push -m "Added backup module and ports" docker/docker-compose.yaml

# 2. Checkout the new branch
git checkout weaviate-1.27

# 3. Add backup volume mount to docker-compose.yaml
# Edit docker/docker-compose.yaml and find the weaviate service
# Add this line under volumes:
#   - ./volumes/weaviate_backups:/var/lib/weaviate/backups

cd docker
docker compose down
docker compose --profile weaviate up -d

# 5. Wait 10-15 seconds for Weaviate to start
sleep 15
```

**After completing the above steps, continue with the cells below:**


---
## PART 3: Restore Backup on Weaviate 1.33.1


### Step 1: Verify Weaviate 1.33.1 is Running


In [19]:
response = requests.get(f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/meta")

if response.status_code == 200:
    meta = response.json()
    print(f"Connected to Weaviate {meta.get('version')}")
    
    if "backup-filesystem" in meta.get("modules", {}):
        print(f"Backup module is enabled")
    else:
        print("WARNING: Backup module NOT enabled")
else:
    print(f"Cannot connect - is Docker running?")


ConnectionError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /v1/meta (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1303971a0>: Failed to establish a new connection: [Errno 61] Connection refused'))

### Step 2: Delete Existing Collections (if any)

Restore will fail if collections already exist. Clean slate:


In [15]:
# Get existing Vector_index collections
response = requests.get(
    f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/schema",
    headers={"Authorization": f"Bearer {WEAVIATE_API_KEY}"}
)

if response.status_code == 200:
    schema = response.json()
    existing = [cls['class'] for cls in schema.get('classes', []) if 'Vector_index' in cls['class']]
    
    if existing:
        print(f"Deleting {len(existing)} existing collections...")
        for col in existing:
            del_response = requests.delete(
                f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/schema/{col}",
                headers={"Authorization": f"Bearer {WEAVIATE_API_KEY}"}
            )
            print(f"  Deleted: {col}" if del_response.status_code == 200 else f"  Failed: {col}")
    else:
        print("No existing collections - ready for restore")
else:
    print("Could not check schema")


Deleting 1 existing collections...
  Deleted: Vector_index_47c65e1f_895e_4ff6_825c_8d2fce9cf011_Node


### Step 2: Restore Backup


In [16]:
# Initiate restore
response = requests.post(
    f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/backups/filesystem/{BACKUP_ID}/restore",
    json={},
    headers={"Authorization": f"Bearer {WEAVIATE_API_KEY}"}
)

if response.status_code in [200, 201]:
    result = response.json()
    print(f"Restore started")
    print(f"  Collections: {result.get('classes')}")
    print(f"  Status: {result.get('status')}")
else:
    print(f"Restore failed: {response.text}")


Restore started
  Collections: ['Vector_index_47c65e1f_895e_4ff6_825c_8d2fce9cf011_Node']
  Status: STARTED


### Step 3: Wait for Restore to Complete


In [17]:
# Poll restore status
print("Waiting for restore to complete...")

for i in range(60):  # Up to 60 seconds
    time.sleep(1)
    
    response = requests.get(
        f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}/v1/backups/filesystem/{BACKUP_ID}/restore",
        headers={"Authorization": f"Bearer {WEAVIATE_API_KEY}"}
    )
    
    if response.status_code == 200:
        status_data = response.json()
        status = status_data.get('status')
        
        if status == 'SUCCESS':
            print(f"\nRestore completed successfully!")
            print(f"  Collections restored: {status_data.get('classes')}")
            break
        elif status == 'FAILED':
            print(f"\nRestore failed!")
            print(f"  Error: {status_data.get('error')}")
            print(json.dumps(status_data, indent=2))
            break
        elif i % 5 == 0:
            print(f"  Status: {status}...")


Waiting for restore to complete...

Restore completed successfully!
  Collections restored: None


---
## PART 4: Run Migration Script

Now run the migration script to fix the schema. This will automatically handle all collections.


In [ ]:
# Run the migration script
%run migrate_weaviate_collections.py


---
## PART 5: Restart Dify and Verify

**In terminal:**
```bash
cd docker
docker compose restart api worker worker_beat
```

**Then test in Dify UI:**
1. Go to http://localhost
2. Open your knowledge base
3. Use "Retrieval Testing"
4. Search for your documents

✓ It should work without the "named vector default" error!


## Setup: Install Dependencies


In [ ]:
%pip install -U weaviate-client requests


# Weaviate Migration and Backup Workflow

This notebook contains all commands for backing up, restoring, and migrating Weaviate collections.

## Setup Environment Variables

In [4]:
import os
import requests
import json
import time

# Weaviate configuration
WEAVIATE_URL = "http://localhost:8080"
AUTH_TOKEN = "WVF5YThaHlkYwhGUSmCRgsX3tD5ngdN8pkih"
BACKUP_ID = "dify-backup-v1-19"
COLLECTION_NAME = "Vector_index_96532f09_a6f9_4492_8afe_7087ed266b01_Node"

# Headers for API requests
headers = {
    "Authorization": f"Bearer {AUTH_TOKEN}",
    "Content-Type": "application/json"
}

## Step 2: Git Operations (Switch Branch)

In [ ]:
# Note: Run these commands in terminal
# !git stash
# !git checkout weaviate-1.27

print("Run the following commands in terminal:")
print("git stash")
print("git checkout weaviate-1.27")

## Step 3: Docker Compose Operations

In [ ]:
# Note: Run these commands in terminal from docker directory
# !docker compose down
# !docker compose --profile weaviate up -d

print("Run the following commands in terminal from docker directory:")
print("cd /Users/dhruvgorasiya/Documents/Weaviate/Integrations/dify/docker")
print("docker compose down")
print("docker compose --profile weaviate up -d")

## Step 4: Restore Backup

In [ ]:
# Restore backup
response = requests.post(
    f"{WEAVIATE_URL}/v1/backups/filesystem/{BACKUP_ID}/restore",
    headers=headers,
    json={}
)

print(f"Restore Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## Step 5: Check Restore Status

In [ ]:
# Wait a bit before checking status
time.sleep(5)

# Check restore status
response = requests.get(
    f"{WEAVIATE_URL}/v1/backups/filesystem/{BACKUP_ID}/restore",
    headers={"Authorization": f"Bearer {AUTH_TOKEN}"}
)

result = response.json()
print(f"Restore Status: {result.get('status', 'Unknown')}")
print(json.dumps(result, indent=2))

## Step 6: Delete Collection (if needed)

In [ ]:
# Delete collection
response = requests.delete(
    f"{WEAVIATE_URL}/v1/schema/{COLLECTION_NAME}",
    headers={"Authorization": f"Bearer {AUTH_TOKEN}"}
)

print(f"Delete Status: {response.status_code}")
if response.status_code == 200:
    print("Collection deleted successfully")
else:
    print(response.text)

## Step 7: Run Migration Script

In [ ]:
# Note: Run migration script from project root
# First setup virtual environment if not already done:
# !cd /Users/dhruvgorasiya/Documents/Weaviate/Integrations/dify
# !python3 -m venv weaviate_migration_env
# !source weaviate_migration_env/bin/activate
# !pip install weaviate-client requests
# !python migrate_weaviate_collections.py

print("Run the following commands in terminal:")
print("cd /Users/dhruvgorasiya/Documents/Weaviate/Integrations/dify")
print("python3 -m venv weaviate_migration_env")
print("source weaviate_migration_env/bin/activate")
print("pip install weaviate-client requests")
print("python migrate_weaviate_collections.py")

## Step 8: Restart Docker Services

In [ ]:
# Note: Run these commands in terminal from docker directory
# !cd docker
# !docker compose restart api worker worker_beat

print("Run the following commands in terminal:")
print("cd /Users/dhruvgorasiya/Documents/Weaviate/Integrations/dify/docker")
print("docker compose restart api worker worker_beat")